# 02 Noise and Detectors

Use this notebook to inspect one noisy-label setting, compare detector scores, and see how cleaning affects one model.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.base import clone

from src.cleaning.cleaning_methods import select_top_k_suspicious
from src.cleaning.detectors import cross_validated_probabilities, scores_from_probabilities
from src.data.load_datasets import load_dataset
from src.data.preprocessing import make_train_val_test_split
from src.evaluation.experiment_runner import run_single_experiment
from src.evaluation.metrics import detection_metrics
from src.models.baselines import get_baseline_models
from src.noise.inject_noise import inject_class_dependent_noise, inject_symmetric_noise

## Choose One Experiment Setting

In [ ]:
DATASET = 'dry_bean'
MODEL = 'random_forest'
NOISE_TYPE = 'symmetric'
NOISE_RATE = 0.2
SEED = 42
CV_FOLDS = 5
DETECTORS = ['combined', 'confidence', 'disagreement', 'margin']

In [ ]:
dataset = load_dataset(DATASET)
X_train, X_val, X_test, y_train, y_val, y_test = make_train_val_test_split(
    dataset.X,
    dataset.y,
    random_state=SEED,
)

if NOISE_TYPE == 'symmetric':
    y_train_noisy, corrupted_mask = inject_symmetric_noise(y_train, NOISE_RATE, random_state=SEED)
elif NOISE_TYPE == 'class_dependent':
    y_train_noisy, corrupted_mask = inject_class_dependent_noise(y_train, NOISE_RATE, random_state=SEED)
else:
    raise ValueError(f'Unknown noise type: {NOISE_TYPE}')

print(f'Dataset: {dataset.name}')
print(f'Train/validation/test: {len(X_train)} / {len(X_val)} / {len(X_test)}')
print(f'Injected noisy labels: {corrupted_mask.sum()} of {len(corrupted_mask)}')

## Detector Metrics

In [ ]:
model = clone(get_baseline_models(random_state=SEED)[MODEL])
cv_proba = cross_validated_probabilities(
    model,
    X_train,
    y_train_noisy,
    cv=CV_FOLDS,
    random_state=SEED,
)

score_frame = pd.DataFrame(index=y_train.index)
metric_rows = []

for detector in DETECTORS:
    scores = scores_from_probabilities(cv_proba, y_train_noisy, detector=detector)
    suspicious_mask = select_top_k_suspicious(scores, fraction=NOISE_RATE)
    score_frame[detector] = scores
    metric_rows.append({'detector': detector, **detection_metrics(corrupted_mask, suspicious_mask)})

detector_metrics = pd.DataFrame(metric_rows).sort_values('det_f1', ascending=False)
display(detector_metrics)

In [ ]:
plt.figure(figsize=(8, 5))
for detector in DETECTORS:
    plt.hist(score_frame[detector], bins=30, alpha=0.45, label=detector)
plt.xlabel('Suspiciousness score')
plt.ylabel('Training samples')
plt.title(f'Detector score distributions: {DATASET}, {NOISE_TYPE}, rate={NOISE_RATE}')
plt.legend()
plt.tight_layout()

## Most Suspicious Samples

In [ ]:
preview = pd.DataFrame(
    {
        'original_index': y_train.index,
        'clean_label': y_train.to_numpy(),
        'noisy_label': y_train_noisy.to_numpy(),
        'was_corrupted': corrupted_mask,
        'combined_score': score_frame['combined'].to_numpy(),
        'confidence_score': score_frame['confidence'].to_numpy(),
        'disagreement_score': score_frame['disagreement'].to_numpy(),
        'margin_score': score_frame['margin'].to_numpy(),
    }
)

display(preview.sort_values('combined_score', ascending=False).head(20))

## One Full Experiment Result

In [ ]:
single_result = run_single_experiment(
    dataset_name=DATASET,
    noise_type=NOISE_TYPE,
    noise_rate=NOISE_RATE,
    model_name=MODEL,
    random_state=SEED,
    cv_folds=CV_FOLDS,
    detector_methods=DETECTORS,
)

display(
    single_result[
        ['condition', 'detector', 'accuracy', 'f1_macro', 'auroc', 'det_precision', 'det_recall', 'det_f1']
    ].sort_values(['condition', 'detector'])
)